# Q1 유지 · 새 Q2 · 이전 Q3 — gold 후보 자동 검증

**새 L4/A100 런타임 → 기존 Secret GITHUB_TOKEN 허용 → 런타임 / 모두 실행 → Drive 승인 → 결과 ZIP 전달.**
박스 그리기·정답 입력은 없습니다. 기존 결과를 읽고 새 폴더에만 저장합니다. GitHub 결과 자동 게시 없음.

dev240을 보고 고른 **InternVL 후보 1개**를 gold246에 적용합니다. Q1 직접 기준선 답 고정, Q2 수정 정의만, Q3 이전 유무→있을 때 위치. 기준선/이전 agent는 미리 정한 비교군이며 gold 점수로 설정을 다시 선택하지 않습니다. 후보의 실제 gold 결과는 아직 없습니다. 기존 gold는 이미 탐색한 세트이므로 새로운 미사용 holdout으로 부르지 않습니다.

기존 gold diffdate 검색 참조와 Q1/Q2 답이 있으면 검증 후 재사용합니다. 없으면 **새 폴더에서 자동 준비**합니다. Q1 최초 기준선 생성은 가능하지만 후보의 Q1 재질문은0회이며 답은 유지합니다. 참고: 초기 기준선이 없으면492호출이 추가됩니다.

Q3는246장의 **별도9구역+없음 보조 문항**입니다. 이미지 라벨로 영역 유무만 채점하며 **위치 정확도는 null**. 기존42문항4보기 Q3와 합치지 않습니다. 결과에서 Q2 결함별 성능, Q3 정상 오경보와 결함 누락, CI를 함께 봅니다.

공유 신규 도구호출984~1,230회 + 누락된 초기 기준선0~492회. 이전 실험 속도를 기준으로 준비·검색 시간을 더해 대략30~70분 여유를 두세요(이번 notebook 실제 시간 미측정).
20단계마다 Drive 저장. 중단 시 **추론 셀 재실행**. 성공 호출은 재사용, 실패/무효 호출은 재시도합니다. 실패 조건은 비교에서 제외되고 전체 완료를 인정하지 않습니다.

출력: MyDrive/lvlm_insplad_results/gold_v2_q23_candidate_v2/.
설정은 저장소 configs/q1_fixed_q2_cues_q3_legacy_gold_v1.json에 미리 고정되어 있습니다. protocol_lock.json은 실행 전 계약이며 성공/선택 파일이 아닙니다. 기존 frozen_config.json을 읽거나 덮어쓰지 않습니다.

실행 코드는 **792c9eac45eafdedda2013cc31ab969b7df7e041** 커밋으로 고정합니다. 최신 main을 실행하지 않으며 fresh/cached 런타임에서 HEAD와 수정 여부를 검사합니다. 기존 PR56 v1 예측이 이미 있으면 GPU 재실행 없이 lvlm_gold_v2_q23_candidate_reaggregate_colab.ipynb를 CPU에서 모두 실행하세요.


In [ ]:
REPO = "arnold6444/lvlm_insplad"
EXECUTION_COMMIT = "792c9eac45eafdedda2013cc31ab969b7df7e041"
QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"
OLD_GOLD_FOLDER = "lvlm_insplad_results/gold_v2_main_rerun"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_q23_candidate_v2"
CHECKPOINT_EVERY = 20
UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
UNSUP_ZIP_PATH = ""
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""


In [ ]:
# Drive 연결 + repo clone + 패키지
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
QA_DIR = DRIVE_ROOT / QA_FOLDER
OLD_GOLD_DIR = DRIVE_ROOT / OLD_GOLD_FOLDER
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
if not RUN_ROOT.name.startswith("gold_v2_q23_candidate"):
    raise ValueError("새 실험 전용 RUN_FOLDER를 사용하세요.")
REPO_DIR = Path("/content/lvlm_q23_candidate_code")

def run(cmd, *, env=None, check=True):
    """Run a command and stream its output into this cell (Colab hides plain subprocess output)."""
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode} (마지막 출력 위 참고)")
    return p.returncode

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
# Fetch exactly the locked object, never a moving branch.
if not REPO_DIR.exists():
    REPO_DIR.mkdir(parents=True)
    run(["git", "init", REPO_DIR])
    run(["git", "-C", REPO_DIR, "remote", "add", "origin", f"https://github.com/{REPO}.git"])
    run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", EXECUTION_COMMIT], env=git_env)
    run(["git", "-C", REPO_DIR, "checkout", "--detach", EXECUTION_COMMIT])
else:
    existing_remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
    if existing_remote != f"https://github.com/{REPO}.git":
        raise RuntimeError("실행 폴더의 저장소가 다릅니다. 새 런타임을 사용하세요.")
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if actual_revision != EXECUTION_COMMIT:
    raise RuntimeError("고정 커밋과 다른 코드입니다. 새 런타임을 사용하세요.")
if subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO_DIR, text=True).strip():
    raise RuntimeError("고정 코드가 수정됐습니다. 새 런타임을 사용하세요.")
del auth, git_env
run([sys.executable,"-m","pip","install","-q","transformers==5.14.0","accelerate==1.14.0",
     "sentencepiece==0.2.2","pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
from verify_gold_revision import verify as verify_gold_revision
verify_gold_revision(REPO_DIR, EXECUTION_COMMIT)
import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime(L4/A100)을 사용하세요.")
if torch.cuda.get_device_properties(0).total_memory < 20 * 1024**3:
    raise RuntimeError("VRAM 20GB 이상 L4/A100이 필요합니다.")


In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음)", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE, UNSUP_ZIP_PATH)
print("SUP:", SUP_ZIP); print("UNSUP:", UNSUP_ZIP)


sys.path.insert(0, str(REPO_DIR / "scripts"))
import run_gold_v2_q23_candidate as candidate
summary = json.loads((QA_DIR / "build_summary.json").read_text(encoding="utf-8"))
if summary.get("quality_status") != "FROZEN" or summary.get("final_master_images") != 246:
    raise RuntimeError("원래 FROZEN gold246 QA가 필요합니다.")
ALL_GOLD_ROWS = candidate.agent.v2.load_gold_qa_v2(QA_DIR)
GOLD_ROWS = candidate.gold_rows(ALL_GOLD_ROWS)
if len(GOLD_ROWS) != 492:
    raise RuntimeError("gold Q1/Q2 문항수가 다릅니다.")
GOLD_SOURCE_HASH = candidate.old.diagnostic.digest(ALL_GOLD_ROWS)
LOCK_PATH = REPO_DIR / "configs/q1_fixed_q2_cues_q3_legacy_gold_v1.json"
LOCK = json.loads(LOCK_PATH.read_text(encoding="utf-8"))
assert LOCK["models"] == ["internvl3_5_8b"]
# This snapshot is created only in the new run folder; guard repeated preparation.
RUN_ROOT.mkdir(parents=True, exist_ok=True)
GOLD_QA = RUN_ROOT / "retrieval_input_qa.jsonl"
if GOLD_QA.is_file() and candidate.agent.read_jsonl(GOLD_QA) != GOLD_ROWS:
    raise RuntimeError("QA가 바뀌었습니다. 기존 결과를 유지하고 새 RUN_FOLDER를 사용하세요.")
if not GOLD_QA.is_file():
    candidate.save_records(GOLD_QA, GOLD_ROWS)

# Existing searched references are read-only; otherwise generate in the new folder.
old_refs = OLD_GOLD_DIR / "refs_gold" / "retrieved_refs.csv"
new_refs_dir = RUN_ROOT / "refs_gold"
REFS_CSV = old_refs if old_refs.is_file() else new_refs_dir / "retrieved_refs.csv"
if not REFS_CSV.is_file():
    run([sys.executable, REPO_DIR / "scripts/build_retrieved_refs.py", "--qa", GOLD_QA,
         "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP, "--out-dir", new_refs_dir])
candidate.read_references(REFS_CSV, GOLD_ROWS)
BASELINE_PREDICTIONS = OLD_GOLD_DIR / "predictions/internvl3_5_8b/normal_ref_retrieved_diffdate.jsonl"
if BASELINE_PREDICTIONS.is_file():
    reused, _ = candidate.cached_baseline(BASELINE_PREDICTIONS, GOLD_ROWS,
        candidate.read_references(REFS_CSV, GOLD_ROWS), "internvl3_5_8b")
    print("기존 gold 기준선 유효 답:", len(reused), "/492")
else:
    print("기존 gold 기준선이 없어 새 실험 폴더에서 초기492답을 생성합니다.")
print("QA: gold246, 기존 Q3:", sum(r["question_type"] == candidate.agent.Q3 for r in ALL_GOLD_ROWS))
print("후보 Q1 재질문0회, 새 Q2 + 이전 Q3, 위치 정확도 null")
print("정상 참조:", REFS_CSV)


In [ ]:
verify_gold_revision(REPO_DIR, EXECUTION_COMMIT)
# 추론 / 자동 채점. 중단되거나 실패가 남으면 이 셀 재실행.
cmd = [sys.executable, REPO_DIR / "scripts/run_gold_v2_q23_candidate.py",
       "--qa-dir", QA_DIR, "--refs-csv", REFS_CSV,
       "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP,
       "--out-dir", RUN_ROOT, "--config", LOCK_PATH,
       "--checkpoint-every", str(CHECKPOINT_EVERY)]
if BASELINE_PREDICTIONS.is_file():
    cmd += ["--baseline-predictions", BASELINE_PREDICTIONS]
run(cmd)
# The original gold QA including four-option Q3 must remain identical.
assert GOLD_SOURCE_HASH == candidate.old.diagnostic.digest(candidate.agent.v2.load_gold_qa_v2(QA_DIR))


In [ ]:
# 자동 결과 확인: 완전한 실행만 성공 인정
pd.set_option("display.max_rows", 100)
status = json.loads((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))
meta = json.loads((RUN_ROOT / "run_metadata.json").read_text(encoding="utf-8"))
display(pd.DataFrame(status["policies"]))
if (not status.get("complete") or not meta.get("complete") or meta.get("row_errors") or meta.get("invalid_answers")
    or len(status["policies"]) != 3 or not all(p["eligible_for_comparison"] for p in status["policies"])):
    raise RuntimeError("실패/무효/누락이 남았습니다. 추론 셀을 재실행하세요. 실패 조건은 비교에서 제외됩니다.")
answers = pd.read_csv(RUN_ROOT / "answers.csv")
q1 = answers[answers.question_type == "anomaly_detection"]
assert len(q1) == meta["images"]*3 and (q1.predicted_choice == q1.cached_baseline_choice).all(), "Q1 답 변경/누락"
assert meta["q1_candidate_calls"] == 0 and meta["q3_location_accuracy"] is None
print("Q1 유지:", len(q1), "행 모두 동일. 후보 재질문0회")
print("신규 기준선:", meta["baseline_calls_this_run"], "신규 도구:", meta["tool_calls_this_run"],
      "저장 성공 재사용:", meta["reused_calls"], "기존 기준선 반입:", meta["imported_baseline_calls"])
lb = pd.read_csv(RUN_ROOT / "leaderboard.csv")
display(lb[["policy", "q1_anomaly_f1", "q1_balanced_accuracy", "q1_precision", "q1_recall",
            "q1_tp", "q1_fp", "q1_tn", "q1_fn", "q2_defect_balanced_accuracy", "q2_defect_macro_f1", "q2_accuracy", "joint_accuracy",
            "false_defect_rate", "q3_presence_balanced_accuracy", "q3_normal_false_region_rate",
            "q3_anomaly_false_none_rate", "q1_q2_q3_presence_joint"]].round(3))
print("자산별 동일 가중 평균")
display(pd.read_csv(RUN_ROOT / "metrics_asset_macro.csv").round(3))
print("결함별 Q2 정답수/지원수")
display(pd.read_csv(RUN_ROOT / "q2_by_defect.csv"))
print("자산별 지원수와 Q3 유무(위치 정확도 아님)")
display(pd.read_csv(RUN_ROOT / "metrics_by_asset.csv").round(3))
print("후보 vs 기준선 / 이전 agent의 사전 고정 비교와 paired CI")
display(pd.read_csv(RUN_ROOT / "significance.csv").round(3))
print("Q3 항상없음/항상영역 대조")
display(pd.read_csv(RUN_ROOT / "q3_constant_baselines.csv").round(3))
print("Q3 위치 정확도 null. 기존42문항4보기 점수와 합치지 않음.")
print("완료. gold에서 설정 재선택 없음. 결과 ZIP을 전달하세요:", RUN_ROOT)


In [ ]:
# 성공/실패 산출물 모두 내려받을 수 있습니다. 실패 ZIP은 완료 결과가 아닙니다.
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = Path("/content/gold_q23_candidate_results.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for p in RUN_ROOT.rglob("*"):
        if p.is_file() and "_images" not in p.relative_to(RUN_ROOT).parts and "_dino_cache" not in p.relative_to(RUN_ROOT).parts:
            z.write(p, p.relative_to(RUN_ROOT))
files.download(str(archive))
